# 6 · A person in the loop

A node that needs a person returns `AskPerson`. Only what reads it waits; the run's state ends `waiting` with every question on it. `compiled.answer(state, answers)` writes the answers in and `compiled.run(state)` goes on.

In [ ]:
from typing import Annotated

from conductor import node
import conductor_nodes
from conductor import AskPerson, CompiledGraph, From, Graph, GraphNode, Param, Result, Static, node
from conductor.widgets import Textarea
from conductor_nodes.types import Text

Decision = Annotated[Text, Result(title="Decision")]


@node("approve", title="Approve", description="Asks a person to approve the proposal", category="review")
def approve(proposal: Annotated[Text, Param(title="Proposal", widget=Textarea())] = Text("")) -> Decision | AskPerson:
    """Asks a person to approve a proposal; the person's answer is the node's output."""
    return AskPerson(prompt="Approve or rewrite the proposal.", defaults={"result": proposal}, widgets={"result": Textarea()})


registry = conductor_nodes.registry(categories=["text"])
registry = registry.extended_with([approve])

## One question

`shout` waits for `approve`; `other` runs in the same leg.

In [ ]:
compiled = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="approve", type="approve", version=1, bindings={"proposal": Static("Ship it")}),
        GraphNode(id="shout", type="text-uppercase", version=1, bindings={"text": From("approve.result")}),
        GraphNode(id="other", type="text-uppercase", version=1, bindings={"text": Static("independent")}),
    ]),
    registry,
)

state = await compiled.run()

print(state.status)
for unit in state.waiting:
    print(unit.id, unit.row, unit.prompt, [(q.name, q.default) for q in unit.questions])
print("ran in this leg:", sorted(compiled.results(state)))

The next leg:

In [ ]:
answered = compiled.answer(state, {"approve": {"result": "Ship it on Monday"}})

started = []
async with compiled.stream(answered) as events:
    async for event in events:
        if event.type == "node_start":
            started.append(event.node_id)
        ended = event

print(ended.state.status)
print(compiled.results(ended.state)["shout"]["result"])
print("started again:", started)

## A question per row

A node on rows asks once per row. The answer is a `Series` on its index naming the rows it answers; a row left out asks again.

In [ ]:
compiled = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="drafts", type="text-split", version=1, bindings={"text": Static("first draft,second draft")}),
        GraphNode(id="approve", type="approve", version=1, bindings={"proposal": From("drafts.result")}),
        GraphNode(id="joined", type="text-join", version=1, bindings={"parts": From("approve.result"), "separator": Static(" | ")}),
    ]),
    registry,
)

state = await compiled.run()
print(state.status, [(unit.row, unit.questions[0].default) for unit in state.waiting])

answers = {unit.id: {"result": f"{unit.questions[0].default}, approved"} for unit in state.waiting}

done = await compiled.run(compiled.answer(state, answers))
print(done.status, compiled.results(done)["joined"]["result"])

From a script:

```python
paused = compiled.run_sync()
done = compiled.run_sync(compiled.answer(paused, answers))
```